# A3 · FunSearch Library Growth v0 (L1 OOV extension)

**Track A · PHYRE W2 · Colab CPU (DeepSeek-V3 API) · wall ≤1h**

When the L1 seed vocabulary (|V|=15 |S|=20 |M|=12 |C|=10 = 57 primitives) cannot explain an OOV mechanism question from the D1 benchmark, an LLM proposes new primitives, validated by held-out rule coverage.

## Protocol (crude FunSearch approximation — 1 generation, LLM as mutator, coverage as fitness)

For each of the 4 OOV questions in D1 (`phyre_L3_007`, `phyre_L4_008`, `phyre_L4_009`, `phyre_L3_010`):

1. **(a) Fit-check** — try to map `ground_truth.mechanism_subgraph` to L1 seed vocab. If ≥1 node uses a V/S/M/C name not in the vocab → OOV triggered.
2. **(b) Proposal** — LLM proposes ≤3 candidate primitive names per category (minimal extensions; reuse existing names via alias when possible).
3. **(c) Validation** — for each candidate, re-score text-match coverage against held-out rule corpus. Compute Δ = candidate appearances beyond what A1 vocab already caught. **Accept if Δ ≥ 2 rules**.
4. **(d) Merge** — accepted primitives are merged into `ontology_v2_seed.json` (N ≥ 58).

## Success criterion
- **≥1** accepted primitive across the 4 OOV questions
- ontology grows from N=57 → N≥58
- coverage lift on ≥2/4 OOV questions (Cell 7)

If 0 accepted → fall back to Δ≥1 threshold (Cell 9 Go/No-Go).

In [ ]:
# ========== 1. setup ==========
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, json, re, time, collections, random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

PHYRE_ROOT  = Path('/content/drive/MyDrive/phyre')
VOCAB_V1    = PHYRE_ROOT / 'data' / 'ontology_v1_seed.json'
VOCAB_V2    = PHYRE_ROOT / 'data' / 'ontology_v2_seed.json'
BENCHMARK   = PHYRE_ROOT / 'data' / 'benchmark' / 'phyre_oracle_v1_draft10.jsonl'
RULES_DIR   = PHYRE_ROOT / 'data' / 'echem_rules' / 'staging'
LOG_GROWTH  = PHYRE_ROOT / 'logs' / 'A3_growth_log.jsonl'
LOG_REVIEW  = PHYRE_ROOT / 'logs' / 'A3_review.md'
for p in [VOCAB_V2.parent, LOG_GROWTH.parent]:
    p.mkdir(parents=True, exist_ok=True)

os.environ['DEEPSEEK_API_KEY'] = userdata.get('DEEPSEEK_API_KEY')
!pip -q install openai
import subprocess
if subprocess.run(['which', 'git'], capture_output=True).returncode != 0:
    !apt-get install -qq -y git

# Load A1 vocab
vocab_v1 = json.loads(VOCAB_V1.read_text(encoding='utf-8'))
n_v1 = sum(len(vocab_v1['vocab'][k]) for k in 'VSMC')
print(f'A1 vocab loaded: |V|={len(vocab_v1["vocab"]["V"])} '
      f'|S|={len(vocab_v1["vocab"]["S"])} '
      f'|M|={len(vocab_v1["vocab"]["M"])} '
      f'|C|={len(vocab_v1["vocab"]["C"])} → N={n_v1}')

# Load D1 benchmark
# Auto-fetch D1 outputs from GitHub repo if not on Drive
GITHUB_REPO = 'https://github.com/wjtwzgl12/reasoning-LLM.git'

def _ensure_d1_on_drive():
    if BENCHMARK.is_file():
        print(f'  ✓ D1 benchmark already on Drive: {BENCHMARK}')
        return
    print(f'  D1 benchmark not on Drive — fetching from GitHub')
    BENCHMARK.parent.mkdir(parents=True, exist_ok=True)
    import subprocess
    tmp = '/tmp/_rllm_d1_fetch'
    subprocess.run(['rm', '-rf', tmp], check=False)
    r = subprocess.run(['git', 'clone', '--depth', '1', GITHUB_REPO, tmp],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print('  ✗ git clone failed:', r.stderr[:300])
        raise SystemExit(
            'Could not auto-fetch D1 outputs. Manual fix:
'
            '  1. Run nb/D1_benchmark_schema.ipynb to produce:
'
            '     - phyre_oracle_schema.json
'
            '     - phyre_oracle_v1_draft10.jsonl
'
            '  2. git add + commit + push to ' + GITHUB_REPO + '
'
            '  3. Re-run this cell')
    src = Path(tmp) / 'pvgap_experiment' / 'data' / 'benchmark'
    if not src.is_dir():
        raise SystemExit(f'Repo missing benchmark dir at {src} — D1 not yet pushed')
    fetched = 0
    for f in src.glob('phyre_oracle*'):
        dst = BENCHMARK.parent / f.name
        dst.write_bytes(f.read_bytes())
        print(f'    fetched {f.name} → {dst}')
        fetched += 1
    if fetched == 0:
        raise SystemExit('Repo has no phyre_oracle_* files — push D1 outputs first')
    print(f'  ✓ fetched {fetched} D1 file(s) to Drive')

_ensure_d1_on_drive()

benchmark = [json.loads(l) for l in open(BENCHMARK, encoding='utf-8') if l.strip()]
print(f'D1 benchmark loaded: {len(benchmark)} questions')

OOV_IDS = {'phyre_L3_007', 'phyre_L4_008', 'phyre_L4_009', 'phyre_L3_010'}
oov_questions = [q for q in benchmark if q.get('qid') in OOV_IDS or q.get('id') in OOV_IDS]
# Fall back: try 'question_id' or first-key id field
if not oov_questions:
    oov_questions = []
    for q in benchmark:
        for k in ('qid', 'id', 'question_id', 'ID'):
            if q.get(k) in OOV_IDS:
                oov_questions.append(q); break
print(f'OOV questions extracted: {len(oov_questions)}')
for q in oov_questions:
    qid = q.get('qid') or q.get('id') or q.get('question_id')
    print(f'  - {qid}')

In [ ]:
# ========== 2. vocab_set + coverage check ==========
def vocab_set(ontology):
    """Return {cat: frozenset(names + aliases)} for fast membership test."""
    out = {}
    for cat in 'VSMC':
        names = set()
        for e in ontology['vocab'][cat]:
            names.add(e['name'].lower().strip())
            for a in e.get('aliases', []) or []:
                names.add(a.lower().strip())
        out[cat] = frozenset(names)
    return out

def _norm(s):
    return re.sub(r'[\s_]+', '-', str(s).strip().lower())

def check_coverage(subgraph, vset):
    """Return dict of missing primitives per category + covered_pct."""
    missing = {f'missing_{k}': [] for k in 'VSMC'}
    total = 0; covered = 0
    nodes = subgraph.get('nodes', []) if isinstance(subgraph, dict) else []
    for node in nodes:
        for k in 'VSMC':
            val = node.get(k)
            if val is None: continue
            vals = val if isinstance(val, list) else [val]
            for v in vals:
                if not v: continue
                total += 1
                vn = _norm(v)
                if vn in vset[k]:
                    covered += 1
                else:
                    missing[f'missing_{k}'].append(vn)
    pct = covered / total if total else 1.0
    return {**missing, 'covered_pct': pct, 'covered': covered, 'total': total}

vset_v1 = vocab_set(vocab_v1)

oov_coverage_before = {}
for q in oov_questions:
    qid = q.get('qid') or q.get('id') or q.get('question_id')
    gt = q.get('ground_truth', {}) or {}
    sg = gt.get('mechanism_subgraph', {}) or {}
    cov = check_coverage(sg, vset_v1)
    oov_coverage_before[qid] = cov
    print(f'\n[{qid}] covered {cov["covered"]}/{cov["total"]} ({cov["covered_pct"]*100:.1f}%)')
    for k in 'VSMC':
        miss = cov[f'missing_{k}']
        if miss: print(f'    missing_{k}: {miss}')

In [ ]:
# ========== 3. LLM proposal prompt ==========
from openai import OpenAI
client = OpenAI(api_key=os.environ['DEEPSEEK_API_KEY'],
                base_url='https://api.deepseek.com')

PROPOSE_SYSTEM = """You are an electrochemistry ontology engineer. The current primitive library has categories V (verbs) S (substrates) M (modifiers) C (conditions). A new mechanism question mentions primitives not in the library. Propose up to 3 new primitives per category that are MINIMAL extensions — reuse existing names via alias when possible. Output STRICT JSON: {"V": [{"name": ..., "def": ..., "aliases": [...]}], "S": [...], "M": [...], "C": [...]}."""

def _names_only(ontology):
    return {k: [e['name'] for e in ontology['vocab'][k]] for k in 'VSMC'}

def build_propose_user(q, cov):
    current_names = _names_only(vocab_v1)
    missing = {k: cov[f'missing_{k}'] for k in 'VSMC'}
    qtext = q.get('question') or q.get('text') or q.get('prompt') or ''
    gt    = q.get('ground_truth', {}) or {}
    sg    = gt.get('mechanism_subgraph', {}) or {}
    body = {
        'current_vocab_names': current_names,
        'missing_from_subgraph': missing,
        'question_text': qtext,
        'ground_truth_subgraph': sg,
        'key_elements': gt.get('key_elements', []),
    }
    return json.dumps(body, ensure_ascii=False, indent=2)

def propose_primitives(q, cov):
    user_msg = build_propose_user(q, cov)
    for attempt in range(3):
        try:
            r = client.chat.completions.create(
                model='deepseek-chat',
                messages=[{'role': 'system', 'content': PROPOSE_SYSTEM},
                          {'role': 'user', 'content': user_msg}],
                response_format={'type': 'json_object'},
                temperature=0.3, max_tokens=800)
            out = json.loads(r.choices[0].message.content)
            # Normalize
            clean = {}
            for k in 'VSMC':
                lst = out.get(k) or []
                cands = []
                for item in lst[:3]:
                    if isinstance(item, dict) and item.get('name'):
                        cands.append({
                            'name': _norm(item['name']),
                            'def': (item.get('def') or '').strip(),
                            'aliases': [_norm(a) for a in (item.get('aliases') or []) if a],
                        })
                    elif isinstance(item, str):
                        cands.append({'name': _norm(item), 'def': '', 'aliases': []})
                clean[k] = cands
            return clean
        except Exception as e:
            if attempt == 2: return {'V':[], 'S':[], 'M':[], 'C':[], '_err': str(e)[:200]}
            time.sleep(2 ** attempt)

print('propose_primitives() ready')

In [ ]:
# ========== 4. run proposals for 4 OOV questions ==========
candidates = []  # (name, cat, source_qid, proposer_def, aliases)
proposals_by_qid = {}

for q in oov_questions:
    qid = q.get('qid') or q.get('id') or q.get('question_id')
    cov = oov_coverage_before[qid]
    prop = propose_primitives(q, cov)
    proposals_by_qid[qid] = prop
    for cat in 'VSMC':
        for item in prop.get(cat, []):
            candidates.append({
                'cat': cat,
                'name': item['name'],
                'def': item['def'],
                'aliases': item['aliases'],
                'source_qid': qid,
            })
    print(f'[{qid}] proposed {sum(len(prop.get(c, [])) for c in "VSMC")} candidates')

# Dedupe by (cat, name) — keep first proposer's def, merge source_qids and aliases
seen = {}
for c in candidates:
    key = (c['cat'], c['name'])
    if key in seen:
        seen[key]['source_qids'].append(c['source_qid'])
        for a in c['aliases']:
            if a not in seen[key]['aliases']: seen[key]['aliases'].append(a)
    else:
        seen[key] = {'cat': c['cat'], 'name': c['name'], 'def': c['def'],
                     'aliases': list(c['aliases']),
                     'source_qids': [c['source_qid']]}
candidates_uniq = list(seen.values())

print(f'\n=== {len(candidates_uniq)} unique candidates ===')
print(f'{"cat":<4}{"name":<32}{"source_qid":<20}def')
print('-' * 100)
for c in candidates_uniq:
    src = ','.join(c['source_qids'])
    print(f'{c["cat"]:<4}{c["name"]:<32}{src:<20}{c["def"][:60]}')

In [ ]:
# ========== 5. coverage-gain validation (text-match on held-out rule corpus) ==========
# Sources combined into one corpus:
#   (1) data/echem_rules/staging/*.jsonl       — 18 raw L1 rules
#   (2) logs/A1_expand.jsonl                   — LLM sub-rule expansion (~288 lines,
#                                                each {parent_rid, parent_source,
#                                                sub_idx, rule: <text>})
#   (3) §9E.1 benchmark (128 entries)          — composed rule_text per A1 cell-3
#                                                (question + structured GT)
#   (4) D1 benchmark question_text + GT        — 10 entries; per-candidate we exclude
#                                                only the source qid, not all 4 OOV
EXPAND_LOG    = PHYRE_ROOT / 'logs' / 'A1_expand.jsonl'

# Hunt for §9E.1 in standard A1 locations
_9E1_PATHS = [
    PHYRE_ROOT / 'data' / 'benchmark' / 'echem_reason_benchmark.jsonl',
    PHYRE_ROOT / 'pvgap_experiment' / 'data' / 'benchmark' / 'echem_reason_benchmark.jsonl',
    Path('/content/phyre-paper1/pvgap_experiment/data/benchmark/echem_reason_benchmark.jsonl'),
    Path('/content/pvgap_experiment/data/benchmark/echem_reason_benchmark.jsonl'),
]
_9E1_FILE = next((p for p in _9E1_PATHS if p.is_file()), None)

# Each rule_text is paired with its source-qid (or None for non-benchmark sources)
# so per-candidate validation can exclude the candidate's own source question.
rule_texts_with_src = []   # list of (text_lower, source_qid_or_None)

# (1) staging
if RULES_DIR.is_dir():
    n0 = len(rule_texts_with_src)
    for f in sorted(RULES_DIR.glob('*.jsonl')):
        for line in open(f, encoding='utf-8'):
            try:
                r = json.loads(line)
                t = None
                for k in ('rule', 'text', 'content', 'statement', 'body'):
                    v = r.get(k)
                    if isinstance(v, str) and v.strip(): t = v; break
                if t is None: t = json.dumps(r, ensure_ascii=False)
                rule_texts_with_src.append((t.lower(), None))
            except Exception:
                pass
    print(f'  (1) staging rules: +{len(rule_texts_with_src) - n0}')

# (2) LLM-expanded sub-rules — schema is {parent_rid, parent_source, sub_idx, rule}
if EXPAND_LOG.is_file():
    n0 = len(rule_texts_with_src)
    for line in open(EXPAND_LOG, encoding='utf-8'):
        try:
            r = json.loads(line)
            t = r.get('rule') or r.get('text') or r.get('statement')
            if isinstance(t, str) and t.strip():
                rule_texts_with_src.append((t.lower(), None))
            # back-compat: also accept old {sub_rules: [...]} schema
            for sub in r.get('sub_rules', []) or []:
                if isinstance(sub, str) and sub.strip():
                    rule_texts_with_src.append((sub.lower(), None))
        except Exception:
            pass
    print(f'  (2) expand log sub-rules: +{len(rule_texts_with_src) - n0}')
else:
    print(f'  (2) {EXPAND_LOG} not found — skipped')

# (3) §9E.1 benchmark (128 entries) — compose same way A1 cell-3 does
def _compose_9e1_text(entry):
    parts = []
    qt = entry.get('question_text') or entry.get('question') or entry.get('prompt') or ''
    if qt: parts.append(qt)
    gt = entry.get('ground_truth', {}) or {}
    if isinstance(gt.get('answer'), str): parts.append(gt['answer'])
    for fld in ('diagnosis', 'degradation_type', 'severity', 'noop_text', 'adversarial_type'):
        v = gt.get(fld)
        if isinstance(v, str) and v.strip(): parts.append(v)
    ki = gt.get('knowledge_ingredients')
    if isinstance(ki, list): parts.extend(str(x) for x in ki)
    elif isinstance(ki, dict):
        parts.extend(f'{k}: {v}' for k,v in ki.items())
    eis = gt.get('eis_features')
    if isinstance(eis, dict):
        parts.extend(f'{k}: {v}' for k,v in eis.items())
    elif isinstance(eis, list):
        parts.extend(str(x) for x in eis)
    stages = gt.get('stages', {})
    if isinstance(stages, dict):
        s3 = stages.get('S3_mechanism_id') or stages.get('S3') or ''
        if s3: parts.append(str(s3))
    return ' . '.join(p for p in parts if p)

if _9E1_FILE is not None:
    n0 = len(rule_texts_with_src)
    for line in open(_9E1_FILE, encoding='utf-8'):
        line = line.strip()
        if not line: continue
        try:
            e = json.loads(line)
            t = _compose_9e1_text(e)
            if t:
                qid = e.get('id') or e.get('qid') or e.get('question_id')
                rule_texts_with_src.append((t.lower(), qid))
        except Exception:
            pass
    print(f'  (3) §9E.1 benchmark ({_9E1_FILE.name}): +{len(rule_texts_with_src) - n0}')
else:
    print(f'  (3) §9E.1 benchmark not found in any of {len(_9E1_PATHS)} locations — skipped')

# (4) D1 benchmark — tag with qid so per-candidate exclusion works
n0 = len(rule_texts_with_src)
for q in benchmark:
    qid = q.get('qid') or q.get('id') or q.get('question_id')
    qtext = q.get('question') or q.get('text') or q.get('prompt') or ''
    if qtext: rule_texts_with_src.append((qtext.lower(), qid))
    gt = q.get('ground_truth', {}) or {}
    sg = gt.get('mechanism_subgraph', {}) or {}
    if sg: rule_texts_with_src.append((json.dumps(sg, ensure_ascii=False).lower(), qid))
    ke = gt.get('key_elements', [])
    if ke: rule_texts_with_src.append((' '.join(str(x) for x in ke).lower(), qid))
print(f'  (4) D1 benchmark: +{len(rule_texts_with_src) - n0}  (from {len(benchmark)} qs)')

print(f'\nloaded {len(rule_texts_with_src)} rule texts total')

def _variants(name):
    n = name.lower().strip()
    return {n, n.replace('-', ' '), n.replace('-', '_'), n.replace(' ', '-')}

# Build baseline vocab variants set per category
vocab_v1_variants = {k: set() for k in 'VSMC'}
for k in 'VSMC':
    for e in vocab_v1['vocab'][k]:
        vocab_v1_variants[k].update(_variants(e['name']))
        for a in e.get('aliases', []) or []:
            vocab_v1_variants[k].update(_variants(a))

# For each candidate: Δ = rules matching candidate that are NOT already matched by
# any v1 primitive in the same category. Per-candidate, exclude texts whose source
# qid equals the candidate's own source_qid (no self-validation).
ACCEPT_THRESHOLD = 2

def delta_for_candidate(cat, name, aliases, source_qids):
    cand_vars = set()
    cand_vars.update(_variants(name))
    for a in aliases: cand_vars.update(_variants(a))
    v1_vars = vocab_v1_variants[cat]
    excluded = set(source_qids or [])
    delta = 0
    for t, src in rule_texts_with_src:
        if src is not None and src in excluded:  # skip self-source texts
            continue
        cand_hit = any(v and v in t for v in cand_vars)
        if not cand_hit: continue
        base_hit = any(v and v in t for v in v1_vars)
        if not base_hit:
            delta += 1
    return delta

print(f'\n{"cat":<4}{"name":<32}{"delta":<8}accept')
print('-' * 60)
for c in candidates_uniq:
    d = delta_for_candidate(c['cat'], c['name'], c['aliases'], c.get('source_qids', []))
    c['delta'] = d
    c['accept'] = bool(d >= ACCEPT_THRESHOLD)
    print(f'{c["cat"]:<4}{c["name"]:<32}{d:<8}{c["accept"]}')

accepted = [c for c in candidates_uniq if c['accept']]
print(f'\n=> {len(accepted)} / {len(candidates_uniq)} accepted at Δ≥{ACCEPT_THRESHOLD}')

In [ ]:
# ========== 6. write ontology_v2_seed.json ==========
# Optional fallback: if 0 accepted, relax to Δ≥1 (flagged in growth log).
if len(accepted) == 0:
    print('[relax] 0 accepted at Δ≥2 — retrying at Δ≥1')
    for c in candidates_uniq:
        c['accept'] = bool(c['delta'] >= 1)
    accepted = [c for c in candidates_uniq if c['accept']]
    relaxed = True
else:
    relaxed = False

vocab_v2 = json.loads(json.dumps(vocab_v1))  # deep copy
vocab_v2['version'] = 'v2_seed'
vocab_v2.setdefault('provenance', {})['A3_growth'] = {
    'timestamp': time.strftime('%Y-%m-%d %H:%M'),
    'accept_threshold': 1 if relaxed else ACCEPT_THRESHOLD,
    'relaxed': relaxed,
    'n_candidates': len(candidates_uniq),
    'n_accepted': len(accepted),
}

for c in accepted:
    entry = {
        'name': c['name'],
        'freq': int(c['delta']),
        'sources': ['A3_growth'],
        'aliases': c['aliases'],
        'def': c['def'],
        'source_qids': c['source_qids'],
    }
    vocab_v2['vocab'][c['cat']].append(entry)

n_v2 = sum(len(vocab_v2['vocab'][k]) for k in 'VSMC')
with open(VOCAB_V2, 'w', encoding='utf-8') as f:
    json.dump(vocab_v2, f, indent=2, ensure_ascii=False)

print(f'ontology grown: N_v1 = {n_v1}  →  N_v2 = {n_v2}   (+{n_v2 - n_v1})')
print(f'wrote {VOCAB_V2}')

In [ ]:
# ========== 7. coverage impact on OOV questions ==========
vset_v2 = vocab_set(vocab_v2)

print(f'{"qid":<20}{"cov_v1":<12}{"cov_v2":<12}Δ')
print('-' * 55)
lifted = 0; deltas = []
for q in oov_questions:
    qid = q.get('qid') or q.get('id') or q.get('question_id')
    sg  = (q.get('ground_truth', {}) or {}).get('mechanism_subgraph', {}) or {}
    cov_v1 = oov_coverage_before[qid]['covered_pct']
    cov_v2 = check_coverage(sg, vset_v2)['covered_pct']
    d = cov_v2 - cov_v1
    deltas.append(d)
    if d > 0: lifted += 1
    print(f'{qid:<20}{cov_v1*100:>6.1f}%    {cov_v2*100:>6.1f}%    {d*100:+.1f}%')

avg_delta = sum(deltas) / max(1, len(deltas))
print(f'\naverage Δ coverage = {avg_delta*100:+.1f}%   (target ≥ +20%)')
print(f'questions with lift ≥0.01 = {lifted} / {len(oov_questions)}   (target ≥2/4)')

In [ ]:
# ========== 8. growth log + human-review markdown ==========
with open(LOG_GROWTH, 'w', encoding='utf-8') as fh:
    for c in candidates_uniq:
        rec = {
            'cat': c['cat'],
            'name': c['name'],
            'def': c['def'],
            'aliases': c['aliases'],
            'source_qids': c['source_qids'],
            'delta': c['delta'],
            'accept': c['accept'],
            'threshold': 1 if relaxed else ACCEPT_THRESHOLD,
        }
        fh.write(json.dumps(rec, ensure_ascii=False) + '\n')
print(f'wrote {LOG_GROWTH}  ({len(candidates_uniq)} records)')

lines = [
    '# A3 FunSearch Growth — human review\n',
    f'Generated {time.strftime("%Y-%m-%d %H:%M")}  ·  threshold Δ≥{1 if relaxed else ACCEPT_THRESHOLD}'
    f'{" (relaxed)" if relaxed else ""}\n',
    f'- candidates proposed: {len(candidates_uniq)}',
    f'- accepted: {len(accepted)}',
    f'- ontology grew: N {n_v1} → {n_v2}',
    '',
    '**Reviewer task**: for each ACCEPTED entry, decide — is it genuinely a new primitive, or an alias we should have merged into an A1 entry? Mark the "verdict" column.',
    '',
    '## Accepted',
    '| cat | name | def | aliases | Δ | source_qid | verdict |',
    '|---|---|---|---|---|---|---|',
]
for c in accepted:
    al = ', '.join(c['aliases'][:3]) + ('...' if len(c['aliases']) > 3 else '')
    sq = ', '.join(c['source_qids'])
    lines.append(f'| {c["cat"]} | `{c["name"]}` | {c["def"]} | {al} | {c["delta"]} | {sq} |  |')

lines += ['', '## Rejected', '| cat | name | def | Δ | source_qid |', '|---|---|---|---|---|']
for c in candidates_uniq:
    if c['accept']: continue
    sq = ', '.join(c['source_qids'])
    lines.append(f'| {c["cat"]} | `{c["name"]}` | {c["def"]} | {c["delta"]} | {sq} |')

LOG_REVIEW.write_text('\n'.join(lines), encoding='utf-8')
print(f'wrote {LOG_REVIEW}')

## 9. Go / No-Go

**Go criteria**:
- ≥1 accepted primitive (N_v2 ≥ 58)
- coverage lift on ≥2/4 OOV questions

**If 0 accepted @ Δ≥2**: Cell 6 auto-relaxes to Δ≥1 and re-flags via `provenance.A3_growth.relaxed=true`.

**If still 0 accepted @ Δ≥1**: concede — "library-growth triggered but held-out validation weak; 9 rule files likely don't cover the modern OOV mechanisms (e.g. anion-exchange, zinc-ion chemistries)." Write this verdict into `A3_review.md` header and plan W5 main experiment with larger rule corpus (target 200+ rules).

**Sign-off**: open `A3_review.md` in Drive, spot-check accepted entries, mark verdict column. Then:
```
git add data/ontology_v2_seed.json logs/A3_growth_log.jsonl logs/A3_review.md
git commit -m 'A3: FunSearch library growth v0 — N=57→{N_v2}'
git tag w2-growth-done
```